# Ligand docking workflow — Cefoxitina contra PBP-6 (E. coli)

Pipeline de docking proteína–ligando pequeño con **AutoDock Vina**, complementario al notebook PyMOL.

- **Diana** — PBP-6 de *E. coli* (UniProt P08506, PDB **3IT9** apo). Es una D-Ala-D-Ala carboxipeptidasa de la familia de las PBPs; blanco clásico de β-lactámicos.
- **Ligando** — cefoxitina (PubChem CID **441199**), cefamicina inhibidor covalente de PBPs.
- **Referencia** — **3ITA** (PBP-6 con ampicilina unida covalentemente) para comparar la pose predicha contra el modo de unión experimental.

Cada sección es independiente. Edita el bloque **Setup** y ejecuta en orden.

## 1. Setup

Variables de entrada y directorio de salida. `LIGAND_SOURCE` acepta CID de PubChem, SMILES o path a un `.sdf`/`.mol2`.

> **Aviso.** Vina hace docking flexible no covalente. Cefoxitina forma realmente acil-enzima con Ser40 — la pose predicha se interpreta como aproximación pre-acilación.

> **Numeración.** UniProt P08506 incluye el péptido señal (1-27). El PDB 3IT9 usa numeración propia: la triada catalítica anotada en UniProt como Ser66/Lys69 corresponde a las posiciones **Ser40/Lys43** del PDB (motivo SXXK), con los motivos auxiliares **Ser106-Gly-Asn** (SXN) y **Lys209-Thr-Gly** (KTG). Estos cuatro residuos definen el bolsillo activo en 3IT9.

In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import Image, display
import pymol_utils as pu
import ligand_utils as lu

RECEPTOR_PDB = "3IT9"
REFERENCE_PDB = "3ITA"
LIGAND_SOURCE = "441199"   # PubChem CID; can also be a SMILES or path
# 3IT9 numbering: SXXK = Ser40-Lys43, SXN at Ser106, KTG at Lys209
POCKET_RESIDUES = [("A", 40), ("A", 43), ("A", 106), ("A", 209)]
RECEPTOR_CHAINS = ["A"]    # 3IT9 has 4 copies (A-D); meeko/Vina runs on one chain
LIGAND_RESN = "UNL"        # residue name meeko/Vina write for the docked ligand
BOX_PADDING = 8.0
EXHAUSTIVENESS = 16
NUM_MODES = 9
OUT = Path("./out_ligand").resolve()
OUT.mkdir(exist_ok=True)

print("Receptor:", RECEPTOR_PDB, "chains", RECEPTOR_CHAINS)
print("Referencia:", REFERENCE_PDB)
print("Ligando:", LIGAND_SOURCE)
print("Salida:", OUT)

## 2. Preparar receptor

Descarga, limpia (sin aguas/iones/alt-conformers) y convierte a PDBQT. Renderiza cartoon y superficie con los residuos del bolsillo activo resaltados.

In [ ]:
rec_raw = pu.fetch_or_load(RECEPTOR_PDB, OUT)
rec_clean = pu.clean_structure(
    rec_raw,
    OUT / f"receptor_{Path(rec_raw).stem}_clean.pdb",
    keep_chains=RECEPTOR_CHAINS,
)
rec_pdbqt = lu.prepare_receptor_pdbqt(rec_clean, OUT / f"receptor_{Path(rec_raw).stem}.pdbqt")

rec_cartoon = pu.render_cartoon(
    rec_clean,
    OUT / "receptor_cartoon.png",
    color_by="ss",
    highlight_residues=POCKET_RESIDUES,
)
rec_surface = pu.render_surface(
    rec_clean,
    OUT / "receptor_surface.png",
)

print("Receptor limpio: ", rec_clean)
print("Receptor PDBQT:  ", rec_pdbqt)
for p in (rec_cartoon, rec_surface):
    display(Image(str(p)))

## 3. Preparar ligando

Carga la cefoxitina desde PubChem, asigna hidrógenos y cargas Gasteiger, y guarda el `.pdbqt` para Vina.

In [ ]:
lig_mol = lu.load_ligand(LIGAND_SOURCE, OUT)
lig_pdbqt = lu.prepare_ligand_pdbqt(lig_mol, OUT / "ligand.pdbqt")

print("Ligando SDF:   ", lig_mol)
print("Ligando PDBQT: ", lig_pdbqt)

In [ ]:
from rdkit import Chem
from rdkit.Chem import AllChem, Descriptors, Draw

mol = Chem.MolFromMolFile(str(lig_mol), removeHs=False)
smiles = Chem.MolToSmiles(Chem.RemoveHs(mol))
metadata = {
    "SMILES": smiles,
    "MW (Da)": round(Descriptors.MolWt(mol), 2),
    "LogP": round(Descriptors.MolLogP(mol), 2),
    "H-bond donors": Descriptors.NumHDonors(mol),
    "H-bond acceptors": Descriptors.NumHAcceptors(mol),
    "Rotatable bonds": AllChem.CalcNumRotatableBonds(mol),
    "Heavy atoms": mol.GetNumHeavyAtoms(),
}
for k, v in metadata.items():
    print(f"{k:>18}: {v}")

display(Draw.MolToImage(Chem.RemoveHs(mol), size=(420, 320)))

## 4. Definir caja y correr Vina

La caja de búsqueda se calcula a partir de los residuos del bolsillo activo en el receptor limpio. Vina explora con `exhaustiveness=EXHAUSTIVENESS` y devuelve hasta `NUM_MODES` poses ordenadas por afinidad.

In [ ]:
box = lu.compute_box(rec_clean, POCKET_RESIDUES, padding=BOX_PADDING)
print("Centro (x, y, z): ", tuple(round(v, 2) for v in box["center"]))
print("Tamano (x, y, z): ", tuple(round(v, 2) for v in box["size"]))

docked_pdbqt = OUT / "docked.pdbqt"
affinities = lu.run_vina(
    receptor_pdbqt=rec_pdbqt,
    ligand_pdbqt=lig_pdbqt,
    box=box,
    out_pdbqt=docked_pdbqt,
    exhaustiveness=EXHAUSTIVENESS,
    num_modes=NUM_MODES,
)

print("\nAfinidades (kcal/mol):")
affinities

## 5. Analizar poses

Separa las poses en archivos individuales, construye un complejo receptor + top pose, lista contactos a < 4 Å y mide la distancia Ser40 Oγ ↔ carbonilo del β-lactámico. Renderiza las 3 mejores poses.

In [ ]:
poses = lu.split_poses(docked_pdbqt, OUT / "poses")
print(f"Poses extraidas: {len(poses)}")
for i, p in enumerate(poses, 1):
    print(f"  pose {i}: {p}")

# Vina writes PDBQT; convert each pose to PDB (tagged with LIGAND_RESN) so the
# complex-building and analysis cells below can read poses[i].with_suffix(".pdb")
from pymol import cmd

for p in poses:
    cmd.reinitialize()
    cmd.load(str(p), "pose", format="pdbqt")
    cmd.alter("pose", f'resn="{LIGAND_RESN}"')
    cmd.sort()
    cmd.save(str(p.with_suffix(".pdb")), "pose")
    cmd.delete("all")

In [ ]:
top_pose_pdb = poses[0].with_suffix(".pdb")
complex_pdb = OUT / "complex_top.pdb"

# concatenate receptor + top pose into a single PDB for visualization
rec_lines = [l for l in Path(rec_clean).read_text().splitlines() if not l.startswith("END")]
pose_lines = [l for l in Path(top_pose_pdb).read_text().splitlines() if l.startswith(("HETATM", "ATOM", "CONECT"))]
complex_pdb.write_text("\n".join(rec_lines + pose_lines + ["END", ""]))
print("Complejo top:", complex_pdb)

In [ ]:
contacts = lu.ligand_contacts(complex_pdb, ligand_resn=LIGAND_RESN, cutoff=4.0)
contacts_csv = pu.save_interface_csv(contacts, OUT / "ligand_contacts.csv")
print(f"Contactos receptor-ligando (<4 A): {len(contacts)}")
print(f"CSV: {contacts_csv}")
contacts

In [ ]:
from pymol import cmd

cmd.reinitialize()
cmd.load(str(complex_pdb), "cx")

# pick the ligand carbonyl carbon closest to Ser40 OG (catalytic Ser in 3IT9)
ser_og = "/cx//A/SER`40/OG"
lig_carbonyl_C = []
cmd.iterate(
    f"resn {LIGAND_RESN} and elem C and bound_to (resn {LIGAND_RESN} and elem O)",
    "lig_carbonyl_C.append((chain, resi, name))",
    space={"lig_carbonyl_C": lig_carbonyl_C},
)

ser_distance = None
if lig_carbonyl_C:
    distances = []
    for chain, resi, name in lig_carbonyl_C:
        sel = f"/cx//{chain}/{LIGAND_RESN}`{resi}/{name}"
        try:
            d = cmd.get_distance(ser_og, sel)
            distances.append((d, name))
        except Exception:
            continue
    if distances:
        distances.sort()
        ser_distance, closest = distances[0]
        print(f"Distancia Ser40 OG -- ligando {closest}: {ser_distance:.2f} A")
    else:
        print("No se pudo medir distancia Ser40 OG -- carbonilo.")
else:
    print("No se identifico un carbono carbonilo en el ligando.")
cmd.delete("all")

In [ ]:
pose_pngs = []
for i, pose in enumerate(poses[:3], 1):
    png = lu.render_pose(
        receptor_pdb=rec_clean,
        pose_pdbqt=pose,
        out_png=OUT / f"pose_{i}.png",
        contact_residues=POCKET_RESIDUES,
    )
    pose_pngs.append(png)
    print(f"pose {i}: {png}")

for p in pose_pngs:
    display(Image(str(p)))

## 6. Comparar contra 3ITA

Alinea el complejo predicho contra 3ITA (PBP-6 + ampicilina) y compara el centroide de la cefoxitina dockeada con el de la ampicilina cristalográfica.

In [ ]:
native_raw = pu.fetch_or_load(REFERENCE_PDB, OUT)
native_clean = pu.clean_structure(native_raw, OUT / f"native_{Path(native_raw).stem}_clean.pdb")

align_info = pu.align_to_native(
    complex_pdb,
    native_clean,
    out_png=OUT / "alignment_vs_3ITA.png",
)
print(f"RMSD modelo vs 3ITA: {align_info['rmsd']:.3f} A sobre {align_info['n_atoms']} atomos")
display(Image(str(align_info["png"])))

In [ ]:
from pymol import cmd
import numpy as np

cmd.reinitialize()
cmd.load(str(complex_pdb), "model")
cmd.load(str(native_clean), "native")
cmd.align("model", "native")

# native covalent ligand in 3ITA is ampicillin (resn AIC) — fall back to any HETATM if absent
native_lig_sel = "native and resn AIC"
if cmd.count_atoms(native_lig_sel) == 0:
    native_lig_sel = "native and hetatm and not resn HOH and polymer.protein not"
    # safer fallback: any non-water heteroatom not part of protein
    native_lig_sel = "native and hetatm and not resn HOH"

docked_coords = np.array(cmd.get_coords(f"model and resn {LIGAND_RESN}"))
native_coords = np.array(cmd.get_coords(native_lig_sel))

if docked_coords is None or native_coords is None or len(docked_coords) == 0 or len(native_coords) == 0:
    print("No se pudieron extraer coordenadas de uno de los ligandos.")
    centroid_distance = None
else:
    docked_centroid = docked_coords.mean(axis=0)
    native_centroid = native_coords.mean(axis=0)
    centroid_distance = float(np.linalg.norm(docked_centroid - native_centroid))
    print(f"Centroide cefoxitina dockeada: {docked_centroid.round(2)}")
    print(f"Centroide ampicilina (3ITA):   {native_centroid.round(2)}")
    print(f"Distancia entre centroides:    {centroid_distance:.2f} A")

cmd.delete("all")

## 7. Resumen

Tabla final de las 3 mejores poses y listado de archivos generados.

In [ ]:
summary_rows = []
for i in range(min(3, len(poses))):
    pose_pdb = poses[i].with_suffix(".pdb")
    pose_complex = OUT / f"complex_pose{i+1}.pdb"
    rec_lines = [l for l in Path(rec_clean).read_text().splitlines() if not l.startswith("END")]
    pose_lines = [l for l in Path(pose_pdb).read_text().splitlines() if l.startswith(("HETATM", "ATOM", "CONECT"))]
    pose_complex.write_text("\n".join(rec_lines + pose_lines + ["END", ""]))

    contacts_i = lu.ligand_contacts(pose_complex, ligand_resn=LIGAND_RESN, cutoff=4.0)
    aff = float(affinities.iloc[i]["affinity"]) if "affinity" in affinities.columns else float(affinities.iloc[i, 0])
    summary_rows.append({
        "mode": i + 1,
        "affinity_kcal_mol": aff,
        "n_contacts": len(contacts_i),
        "ser40_distance_A": ser_distance if i == 0 else None,
    })

summary = pd.DataFrame(summary_rows)
summary_csv = OUT / "summary_top3.csv"
summary.to_csv(summary_csv, index=False)
print("Top 3 poses:")
print(summary.to_string(index=False))
print()

print("Archivos generados:")
for p in sorted(OUT.glob("*")):
    if p.suffix in {".png", ".csv", ".pdb", ".pdbqt", ".sdf", ".log"}:
        print(f"  {p}")